# RAG Notebook: Pinecone + Groq

- Retrieves context from Pinecone
- Calls Groq `/chat/completions`
- Returns answer + sources
- Uses isolated namespace `notebook-demo` for demo records

In [ ]:
import importlib.util
import subprocess
import sys

def ensure_package(module_name: str, pip_name: str) -> None:
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

ensure_package("pinecone", "pinecone>=5")
ensure_package("dotenv", "python-dotenv>=1")
ensure_package("requests", "requests>=2")

print("Dependency check complete.")


In [ ]:
import os
import sys
import uuid
from getpass import getpass
from typing import Any, Dict, List

import requests
from dotenv import load_dotenv

try:
    from pinecone import Pinecone
except Exception as exc:
    Pinecone = None
    print(f"Pinecone import failed: {exc}")

IN_COLAB = 'google.colab' in sys.modules

def first_non_empty(*values: str, default: str = "") -> str:
    for value in values:
        if isinstance(value, str) and value.strip():
            return value.strip()
    return default

def read_secret(name: str, prompt_if_missing: bool = True) -> str:
    value = os.getenv(name, "").strip()
    if value:
        return value

    if IN_COLAB:
        # Optional: use Colab Secrets (key icon in left sidebar)
        try:
            from google.colab import userdata  # type: ignore
            value = (userdata.get(name) or "").strip()
            if value:
                return value
        except Exception:
            pass

    if prompt_if_missing:
        try:
            return getpass(f"Enter {name} (leave blank to skip): ").strip()
        except Exception:
            return ""
    return ""

# Local .env loading (harmless on Colab if files do not exist).
load_dotenv()
load_dotenv('../App/.env', override=False)

PINECONE_API_KEY = first_non_empty(
    read_secret("PINECONE_API_KEY", prompt_if_missing=False),
    os.getenv("VITE_PINECONE_API_KEY", ""),
)
if not PINECONE_API_KEY and IN_COLAB:
    PINECONE_API_KEY = read_secret("PINECONE_API_KEY", prompt_if_missing=True)

GROQ_API_KEY = first_non_empty(
    read_secret("GROQ_API_KEY", prompt_if_missing=False),
    os.getenv("VITE_GROQ_API_KEY", ""),
)
if not GROQ_API_KEY and IN_COLAB:
    GROQ_API_KEY = read_secret("GROQ_API_KEY", prompt_if_missing=True)

PINECONE_INDEX_NAME = first_non_empty(
    os.getenv("PINECONE_INDEX_NAME", ""),
    os.getenv("VITE_PINECONE_INDEX_NAME", ""),
    default="ORCA",
)
PINECONE_NAMESPACE = first_non_empty(
    os.getenv("PINECONE_NAMESPACE", ""),
    os.getenv("VITE_PINECONE_NAMESPACE", ""),
    default="notebook-demo",
)

GROQ_BASE_URL = first_non_empty(
    os.getenv("GROQ_BASE_URL", ""),
    os.getenv("VITE_GROQ_BASE_URL", ""),
    default="https://api.groq.com/openai/v1",
).rstrip("/")
GROQ_CHAT_MODEL = first_non_empty(
    os.getenv("GROQ_CHAT_MODEL", ""),
    os.getenv("VITE_GROQ_CHAT_MODEL", ""),
    default="openai/gpt-oss-20b",
)

print({
    "in_colab": IN_COLAB,
    "pinecone_index": PINECONE_INDEX_NAME,
    "pinecone_namespace": PINECONE_NAMESPACE,
    "has_pinecone_key": bool(PINECONE_API_KEY),
    "groq_base_url": GROQ_BASE_URL,
    "groq_model": GROQ_CHAT_MODEL,
    "has_groq_key": bool(GROQ_API_KEY),
})


Pinecone import failed: No module named 'pinecone'
{'pinecone_index': 'ORCA', 'pinecone_namespace': 'notebook-demo', 'has_pinecone_key': False, 'groq_base_url': 'https://api.groq.com/openai/v1', 'groq_model': 'openai/gpt-oss-20b', 'has_groq_key': False}


In [3]:
def get_pinecone_index():
    if Pinecone is None:
        print("Pinecone SDK unavailable: running in mock-safe mode.")
        return None
    if not PINECONE_API_KEY:
        print("PINECONE_API_KEY missing: running in mock-safe mode.")
        return None

    try:
        pc = Pinecone(api_key=PINECONE_API_KEY)
        return pc.Index(PINECONE_INDEX_NAME)
    except Exception as exc:
        print(f"Could not open Pinecone index '{PINECONE_INDEX_NAME}': {exc}")
        print("Continuing in mock-safe mode.")
        return None

index = get_pinecone_index()
print("Index ready:", index is not None)


Pinecone SDK unavailable: running in mock-safe mode.
Index ready: False


In [4]:
DEMO_RECORD_IDS: List[str] = []

def upsert_demo_records(idx) -> None:
    global DEMO_RECORD_IDS
    if idx is None:
        print('Skipping upsert (no live Pinecone index).')
        return

    session = uuid.uuid4().hex[:8]
    records = [
        {
            '_id': f'notebook-demo-{session}-1',
            'chunk_text': 'CyberBase uses Pinecone retrieval to ground answers from indexed chunks.',
            'source': 'notebook-demo.md',
            'title': 'RAG Demo Context 1'
        },
        {
            '_id': f'notebook-demo-{session}-2',
            'chunk_text': 'When no relevant chunks are found, the app can fall back to a direct Groq answer.',
            'source': 'notebook-demo.md',
            'title': 'RAG Demo Context 2'
        },
        {
            '_id': f'notebook-demo-{session}-3',
            'chunk_text': 'The chat UI reports returned source identifiers along with the model answer.',
            'source': 'notebook-demo.md',
            'title': 'RAG Demo Context 3'
        },
    ]

    DEMO_RECORD_IDS = [r['_id'] for r in records]
    idx.upsert_records(namespace=PINECONE_NAMESPACE, records=records)
    print(f'Upserted {len(records)} demo records into namespace: {PINECONE_NAMESPACE}')

upsert_demo_records(index)


Skipping upsert (no live Pinecone index).


In [5]:
def search_context(query: str, top_k: int = 4) -> List[Dict[str, Any]]:
    if index is None:
        return [
            {'source': 'mock-source-1', 'snippet': 'Mock snippet about Pinecone search and sources.'},
            {'source': 'mock-source-2', 'snippet': 'Mock snippet about Groq completion fallback behavior.'},
        ]

    response = index.search(
        namespace=PINECONE_NAMESPACE,
        query={
            'top_k': top_k,
            'inputs': {'text': query},
        },
        fields=['chunk_text', 'text', 'source', 'title', 'path', 'url'],
    )

    hits = []
    for hit in getattr(response, 'result', {}).get('hits', []):
        fields = hit.get('fields', {})
        snippet = fields.get('chunk_text') or fields.get('text') or 'No snippet'
        source = fields.get('source') or fields.get('url') or fields.get('path') or fields.get('title') or hit.get('_id') or 'unknown-source'
        hits.append({'source': source, 'snippet': snippet})
    return hits

hits = search_context('How does this project use RAG?')
print('Retrieved hits:', len(hits))
for h in hits[:3]:
    print('-', h['source'])


Retrieved hits: 2
- mock-source-1
- mock-source-2


In [6]:
def answer_with_groq(question: str, context_hits: List[Dict[str, Any]]) -> str:
    context = '\n\n'.join([f"Source {i+1} ({h['source']}):\n{h['snippet']}" for i, h in enumerate(context_hits)])

    if not GROQ_API_KEY:
        return (
            '[MOCK ANSWER] GROQ_API_KEY is missing. '
            'Set GROQ_API_KEY to run live completion.\n\n'
            f'Question: {question}\n'
            f'Context count: {len(context_hits)}'
        )

    response = requests.post(
        f"{GROQ_BASE_URL}/chat/completions",
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {GROQ_API_KEY}',
        },
        json={
            'model': GROQ_CHAT_MODEL,
            'temperature': 0.2,
            'messages': [
                {
                    'role': 'system',
                    'content': 'You are a security assistant. Answer only from the provided context. If context is insufficient, say that clearly.'
                },
                {
                    'role': 'user',
                    'content': f'Question: {question}\n\nContext:\n{context}'
                }
            ]
        },
        timeout=60,
    )
    response.raise_for_status()
    payload = response.json()
    return payload.get('choices', [{}])[0].get('message', {}).get('content', 'No answer returned.').strip()

question = "Explain this project's RAG flow in three bullet points."
answer = answer_with_groq(question, hits)
print(answer)


[MOCK ANSWER] GROQ_API_KEY is missing. Set GROQ_API_KEY to run live completion.

Question: Explain this project's RAG flow in three bullet points.
Context count: 2


In [ ]:
def cleanup_demo_records(idx) -> None:
    if idx is None:
        print('Skipping cleanup (no live Pinecone index).')
        return
    if not DEMO_RECORD_IDS:
        print('No demo record ids to delete in this session.')
        return

    idx.delete(namespace=PINECONE_NAMESPACE, ids=DEMO_RECORD_IDS)
    print(f'Deleted {len(DEMO_RECORD_IDS)} demo records from namespace: {PINECONE_NAMESPACE}')

# Run when you want to clean up notebook demo data:
# cleanup_demo_records(index)
